In [1]:
import numpy as np

## Ridge Regression: 
####       it has closed form solution,
####       The solution is obtained by minimizing 
###        RSS(weights,alpha)= MSE+ alpha*(sum of squared weights)

In [2]:
class Ridge:
    def __init__(self,alpha):
        self.weights=None
        self.alpha=alpha
    def predict(self,X):
        X=np.concatenate((np.ones((X.shape[0],1)),X),axis=1)
        prediction=X@self.weights
        return prediction
    def fit(self,X,y):
        X=np.concatenate((np.ones((X.shape[0],1)),X),axis=1)
        a=X.T@X+self.alpha*np.identity(X.shape[1])
        b=X.T@y
        self.weights=np.linalg.inv(a)@b
    def MSE(self,y,y_hat):
        mse=0
        for r,s in zip(y,y_hat):
            mse+=(r-s)**2
        return float(mse)
    def __repr__(self,):
        return f"Ridge Regressor :\n -> Alpha = {self.alpha}\n -> Weights = {self.weights} "

In [3]:
model=Ridge(3)
X=np.array(
    [[2,4,6],
    [3,6,9],
    [4,8,12]]
)
y=np.array([2,3,4])
model.fit(X,y)

In [4]:
model.predict(np.array([[5,10,15]]))

array([4.95227273])

In [5]:
lam=np.arange(0.25,10,0.25)
best_model=None
best_mse=float('inf')
for l in lam:
    model=Ridge(l)
    model.fit(X,y)
    y_hat=model.predict(X)
    mse=model.MSE(y,y_hat)
    print(model)
    if mse<best_mse:
        best_model=model
        best_mse=mse
print("*"*60)
print(f"Best model is \n {best_model}")
print("*"*60)

Ridge Regressor :
 -> Alpha = 0.25
 -> Weights = [0.01207648 0.07111707 0.14223415 0.21335122] 
Ridge Regressor :
 -> Alpha = 0.5
 -> Weights = [0.01558442 0.07099567 0.14199134 0.21298701] 
Ridge Regressor :
 -> Alpha = 0.75
 -> Weights = [0.01724964 0.07091519 0.14183038 0.21274557] 
Ridge Regressor :
 -> Alpha = 1.0
 -> Weights = [0.01821862 0.0708502  0.1417004  0.21255061] 
Ridge Regressor :
 -> Alpha = 1.25
 -> Weights = [0.01885014 0.07079275 0.14158551 0.21237826] 
Ridge Regressor :
 -> Alpha = 1.5
 -> Weights = [0.0192926  0.07073955 0.1414791  0.21221865] 
Ridge Regressor :
 -> Alpha = 1.75
 -> Weights = [0.01961853 0.07068898 0.14137797 0.21206695] 
Ridge Regressor :
 -> Alpha = 2.0
 -> Weights = [0.01986755 0.07064018 0.14128035 0.21192053] 
Ridge Regressor :
 -> Alpha = 2.25
 -> Weights = [0.02006316 0.07059261 0.14118521 0.21177782] 
Ridge Regressor :
 -> Alpha = 2.5
 -> Weights = [0.02022018 0.07054594 0.14109189 0.21163783] 
Ridge Regressor :
 -> Alpha = 2.75
 -> Weight

## LASSO Regression :
#### Lasso doesn't have any closed form solutio.
#### We continously reweight such that it minimizes the loss function.
#### For lasso the loss function is
###   RSS(weights,alpha)=MSE+alpha*(sum of absolute values of weights)
###   $$J(w) = \text{MSE} + \alpha \sum_{j=1}^{d} |w_j|$$
#### Lasso can converge a weight to zero, that means it can also identify the likely unnecessary variables

## Approach 1: Subgradient Descent

### Mathematical Overview
### Because $|w|$ has a sharp corner at zero, we use the subgradient of the penalty term, which translates directly to the element-wise sign of the weights:
## $$\nabla \Omega(w) = \alpha \cdot \text{sign}(w)$$

### Implementation

In [6]:
class LassoSubgradientDescent:
    def __init__(self,alpha):
        self.alpha=alpha
        self.weights=None
    def predict(self,X):
        X=np.concatenate((np.ones((X.shape[0],1)),X),axis=1)
        prediction=X@self.weights
        return prediction
    def MSE(self,y,y_hat):
        mse=0
        for r,s in zip(y,y_hat):
            mse+=(r-s)**2
        return mse
    def fit(self,X,y,max_iter,learning_rate):
        self.weights=np.random.rand(X.shape[1]+1)
        
        X_expanded=X=np.concatenate((np.ones((X.shape[0],1)),X),axis=1)
        for _ in range(max_iter):
            cost=self.MSE(y,y_hat)+self.alpha*np.sum(np.abs(self.weights))
            p=X_expanded@self.weights
            w_grad_mse=(-2*X_expanded.T@(y-p))/len(y)
            w_grad_penality=self.alpha*np.sign(self.weights)
            grad=w_grad_mse+w_grad_penality
            self.weights-=learning_rate*grad
            print(f"Iternation {_+1} | Loss = {cost}")
            
    def __repr__(self,):
        return f"Lasso Regressor(SubgradientDescent) :\n -> Alpha = {self.alpha}\n -> Weights = {self.weights} "

In [7]:
X_test = np.array([
    [1.0, 2.0, 3.0],
    [2.0, 4.0, 5.0],
    [3.0, 5.0, 1.0],
    [4.0, 1.0, 2.0]
])

# True relationship: y = 5 + 2*x1 + 0*x2 - 3*x3
y_test = np.array([0.0, 0.0, 8.0, 7.0]) 
model = LassoSubgradientDescent(alpha=0.1)
model.fit(X_test, y_test, max_iter=30, learning_rate=0.0001)

Iternation 1 | Loss = 29.424915721739364
Iternation 2 | Loss = 29.424611386737173
Iternation 3 | Loss = 29.424308657684502
Iternation 4 | Loss = 29.4240075264691
Iternation 5 | Loss = 29.423707985019558
Iternation 6 | Loss = 29.42341002530508
Iternation 7 | Loss = 29.4231136393353
Iternation 8 | Loss = 29.42281881916005
Iternation 9 | Loss = 29.422525556869193
Iternation 10 | Loss = 29.422233844592398
Iternation 11 | Loss = 29.421943674498948
Iternation 12 | Loss = 29.421655038797528
Iternation 13 | Loss = 29.421367929736057
Iternation 14 | Loss = 29.421082339601448
Iternation 15 | Loss = 29.420798260719454
Iternation 16 | Loss = 29.420515685454447
Iternation 17 | Loss = 29.42023460620922
Iternation 18 | Loss = 29.41995501542482
Iternation 19 | Loss = 29.419676905580328
Iternation 20 | Loss = 29.419400269192682
Iternation 21 | Loss = 29.419125098816487
Iternation 22 | Loss = 29.418851387043816
Iternation 23 | Loss = 29.418579126504028
Iternation 24 | Loss = 29.418308309863583
Iternatio

In [8]:
print(model)

Lasso Regressor(SubgradientDescent) :
 -> Alpha = 0.1
 -> Weights = [0.10617844 0.65206835 0.892679   0.14404274] 


In [9]:
lam=np.arange(0.25,3,0.25)
best_model=None
best_cost=float('inf')
for l in lam:
    model=LassoSubgradientDescent(l)
    model.fit(X,y,max_iter=100,learning_rate=0.001)
    print(model)
    y_hat=model.predict(X)
    cost=model.MSE(y,y_hat)+model.alpha*np.sum(np.abs(model.weights))
    if cost<best_cost:
        best_model=model
        best_cost=cost
print("*"*60)
print(f"Best model is \n {best_model}")
print(f"Cost value of best_model is {best_cost}")
print("*"*60)

Iternation 1 | Loss = 0.5452732069391153
Iternation 2 | Loss = 0.43559034208523756
Iternation 3 | Loss = 0.3598924829549276
Iternation 4 | Loss = 0.30750061504411924
Iternation 5 | Loss = 0.2693575502370294
Iternation 6 | Loss = 0.24158006985044617
Iternation 7 | Loss = 0.2213432475401309
Iternation 8 | Loss = 0.2065920332298889
Iternation 9 | Loss = 0.1958314389797332
Iternation 10 | Loss = 0.18797390543194775
Iternation 11 | Loss = 0.18222826543053833
Iternation 12 | Loss = 0.17801896834977887
Iternation 13 | Loss = 0.17492731819053214
Iternation 14 | Loss = 0.1726487260392901
Iternation 15 | Loss = 0.17096161250080938
Iternation 16 | Loss = 0.16970478514078402
Iternation 17 | Loss = 0.16876098124679395
Iternation 18 | Loss = 0.16804489567532166
Iternation 19 | Loss = 0.16749447146577273
Iternation 20 | Loss = 0.1670645640206574
Iternation 21 | Loss = 0.16672233198471048
Iternation 22 | Loss = 0.16644388424624346
Iternation 23 | Loss = 0.16621184073013945
Iternation 24 | Loss = 0.166

## Approach 2: Coordinate Descent (Analytical Selection)

### Mathematical Overview
#### Instead of updating all weights simultaneously, Coordinate Descent optimizes **one weight at a time** while holding all others constant. It calculates the partial residual $\rho_j$ (the error left over excluding feature $j$) and applies the **Soft-Thresholding Operator**:

## $$S(\rho_j, \alpha) = \text{sign}(\rho_j) \max(|\rho_j| - \alpha, 0)$$

#### This forces uninformative feature weights to snap cleanly to exactly `0.0`.

### Update rule:
$$
\beta_k =
\begin{cases} 
\frac{\sum_{i=1}^n r_i^{(k)} x_{ik} - \lambda}{\sum_{i=1}^n x_{ik}^2} & \text{if } \sum_{i=1}^n r_i^{(k)} x_{ik} > \lambda \\ 
\frac{\sum_{i=1}^n r_i^{(k)} x_{ik} + \lambda}{\sum_{i=1}^n x_{ik}^2} & \text{if } \sum_{i=1}^n r_i^{(k)} x_{ik} < -\lambda \\ 
0 & \text{if } -\lambda \le \sum_{i=1}^n r_i^{(k)} x_{ik} \le \lambda 
\end{cases}
$$

#### where $$ r_i^{(k)} = y_i - \sum_{j \neq k} x_{ij}\beta_j $$
$$r_i^{(k)} = (y_i - \hat{y}_i) + x_{ik}\beta_k$$

### Implementation

In [10]:
class LassoCoordinateDescent:
    def __init__(self,alpha):
        self.alpha=alpha
        self.weights=None
    def predict(self,X):
        X=np.concatenate((np.ones((X.shape[0],1)),X),axis=1)
        prediction=X@self.weights
        return prediction
    def MSE(self,y,y_hat):
        mse=0
        for r,s in zip(y,y_hat):
            mse+=(r-s)**2
        return mse
    def r(self, X_expanded, y, y_hat, k):
        return (y - y_hat) + X_expanded[:, k] * self.weights[k]
    def update_rule(self,X,y,k):
        rtimesx=0
        x_expanded=np.concatenate((np.ones((X.shape[0],1)),X),axis=1)
        y_hat=x_expanded@self.weights
        for r,s in zip(self.r(x_expanded,y,y_hat,k+1),x_expanded[:,k+1]):
            rtimesx+=r*s
        if rtimesx>self.alpha:
            update_value=(rtimesx-self.alpha)/np.sum(x_expanded[:,k+1]**2)
        elif rtimesx<-self.alpha:
            update_value=(rtimesx+self.alpha)/np.sum(x_expanded[:,k+1]**2)
        else:
            update_value=0
        return update_value
            
    def fit(self,X,y,max_iter):
        samples,features=X.shape
        self.weights=np.random.rand(features+1)
        for _ in range(max_iter):
            y_hat_features = X @ self.weights[1:]
            self.weights[0] = np.mean(y - y_hat_features)
            for k in range(features):
                update_value=self.update_rule(X,y,k)
                self.weights[k+1]=update_value
            y_pred=self.predict(X)
            print(f"Iteration : {_+1} | Loss : {self.MSE(y,y_pred)}")
    def __repr__(self,):
        return f"Lasso Regressor(Coordinate Descent) :\n -> Alpha = {self.alpha}\n -> Weights = {self.weights} "
            
        
        

In [11]:
X_test = np.array([
    [1.0, 2.0, 3.0],
    [2.0, 4.0, 5.0],
    [3.0, 5.0, 1.0],
    [4.0, 1.0, 2.0]
])

# True relationship: y = 5 + 2*x1 + 0*x2 - 3*x3
y_test = np.array([0.0, 0.0, 8.0, 7.0]) 
model = LassoCoordinateDescent(alpha=0.1)
model.fit(X_test, y_test, max_iter=30)

Iteration : 1 | Loss : 33.74936148371108
Iteration : 2 | Loss : 16.705533773134896
Iteration : 3 | Loss : 8.233671965423758
Iteration : 4 | Loss : 4.045525353434531
Iteration : 5 | Loss : 2.0000673266759397
Iteration : 6 | Loss : 1.0057488850057588
Iteration : 7 | Loss : 0.5197050768407941
Iteration : 8 | Loss : 0.2778675219990888
Iteration : 9 | Loss : 0.15396475973460236
Iteration : 10 | Loss : 0.08813735901249785
Iteration : 11 | Loss : 0.051890762400548136
Iteration : 12 | Loss : 0.031372995704262066
Iteration : 13 | Loss : 0.01958473217345378
Iteration : 14 | Loss : 0.01280520487914435
Iteration : 15 | Loss : 0.008952589549178947
Iteration : 16 | Loss : 0.006814687984445771
Iteration : 17 | Loss : 0.005669658283548714
Iteration : 18 | Loss : 0.0050854725440853754
Iteration : 19 | Loss : 0.004805949444758204
Iteration : 20 | Loss : 0.00468233925442869
Iteration : 21 | Loss : 0.004631025579648075
Iteration : 22 | Loss : 0.004607179451997
Iteration : 23 | Loss : 0.004588471578633491


In [12]:
y_pred=model.predict(X_test)
print(model.MSE(y_test,y_pred))
print(model)


0.004304793212341809
Lasso Regressor(Coordinate Descent) :
 -> Alpha = 0.1
 -> Weights = [ 1.51095159  1.97438488  0.38611975 -1.40122195] 


In [13]:
lam=np.arange(0.25,3,0.25)
best_model=None
best_cost=float('inf')
for l in lam:
    model=LassoCoordinateDescent(l)
    model.fit(X,y,max_iter=30)
    print(model)
    y_hat=model.predict(X)
    cost=model.MSE(y,y_hat)+model.alpha*np.sum(np.abs(model.weights))
    if cost<best_cost:
        best_model=model
        best_cost=cost
print("*"*60)
print(f"Best model is \n {best_model}")
print(f"Cost value of best_model is {best_cost}")
print("*"*60)

Iteration : 1 | Loss : 7.01153474522072
Iteration : 2 | Loss : 6.058488695264422
Iteration : 3 | Loss : 5.233726339642984
Iteration : 4 | Loss : 4.520069594061001
Iteration : 5 | Loss : 3.9026345399553763
Iteration : 6 | Loss : 3.3685254526974644
Iteration : 7 | Loss : 2.906569608336319
Iteration : 8 | Loss : 2.5070874360844853
Iteration : 9 | Loss : 2.1616933074060647
Iteration : 10 | Loss : 1.8631228798420585
Iteration : 11 | Loss : 1.6050834574365018
Iteration : 12 | Loss : 1.382124300935761
Iteration : 13 | Loss : 1.1895242294676422
Iteration : 14 | Loss : 1.0231942095270978
Iteration : 15 | Loss : 0.8795929340485603
Iteration : 16 | Loss : 0.7556536604131867
Iteration : 17 | Loss : 0.6487208068677427
Iteration : 18 | Loss : 0.556495006740818
Iteration : 19 | Loss : 0.4769854931227753
Iteration : 20 | Loss : 0.40846883687505503
Iteration : 21 | Loss : 0.3494531910262655
Iteration : 22 | Loss : 0.29864730746179735
Iteration : 23 | Loss : 0.25493368963027147
Iteration : 24 | Loss : 0